# PaddleOCR-VL Fine-Tuning with cord-v2 Dataset

Fine-tune PaddleOCR-VL-1.5 on the naver-clova-ix/cord-v2 receipt OCR dataset using SFTTrainer.

In [ ]:
%pip install -q "transformers>=4.55.0,<5.0.0" datasets trl peft accelerate bitsandbytes Pillow einops protobuf

import os
os.environ["HF_HOME"] = "/dev/shm/hf_cache"

In [ ]:
import json

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault("TRACKIO_SPACE_ID", "trl-trackio")

## Configuration

In [ ]:
MODEL_NAME = "PaddlePaddle/PaddleOCR-VL-1.5"
DATASET_NAME = "naver-clova-ix/cord-v2"
OUTPUT_DIR = "PaddleOCR-VL-finetuned"
OCR_PROMPT = "OCR: Parse this receipt into JSON"

# Training hyperparameters
LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 10
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2
MAX_STEPS = -1  # Set to a positive number to limit training steps (e.g., 1 for testing)

# LoRA configuration (set USE_LORA = False for full fine-tuning)
USE_LORA = True
LORA_R = 64
LORA_ALPHA = 32
LORA_TARGET_MODULES = "all-linear"

## Dataset Formatting

In [ ]:
def format_cord_v2(samples: dict[str, any]) -> dict[str, list]:
    """Format cord-v2 samples into chat messages with JSON targets for SFTTrainer."""
    formatted_samples = {"messages": []}
    for i in range(len(samples["image"])):
        image = samples["image"][i]
        if image.mode != "RGB":
            image = image.convert("RGB")

        gt = json.loads(samples["ground_truth"][i])
        gt_parse = gt.get("gt_parse", {})
        target_text = json.dumps(gt_parse, ensure_ascii=False)

        formatted_samples["messages"].append(
            [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": OCR_PROMPT},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": target_text}],
                },
            ]
        )
    return formatted_samples

## Load Dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
dataset = dataset.map(format_cord_v2, batched=True, batch_size=32, num_proc=32)
print(dataset)

## Load Model

In [ ]:
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Base Model Diagnostic (before training)

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

test_image = dataset["test"][0]["image"].convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image", "image": test_image},
    {"type": "text", "text": OCR_PROMPT},
]}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True,
    tokenize=True, return_dict=True, return_tensors="pt",
).to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512)
print("Base model output (before training):")
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1]))

## Training Configuration

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_strategy="epoch",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    use_cpu=not torch.cuda.is_available(),
)

peft_config = None
if USE_LORA:
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        task_type="CAUSAL_LM",
    )
    print(f"LoRA enabled: r={LORA_R}, alpha={LORA_ALPHA}")

print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=processor,
    peft_config=peft_config,
)

trainer.train()

## Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## Inference

In [ ]:
from transformers import AutoProcessor

# Load processor from base model
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Use a test image from the dataset
test_sample = dataset["test"][0]
test_image = test_sample["image"]
if test_image.mode != "RGB":
    test_image = test_image.convert("RGB")

# Prepare input
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {"type": "text", "text": OCR_PROMPT},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(trainer.model.device)

# Generate
trainer.model.eval()
with torch.no_grad():
    outputs = trainer.model.generate(**inputs, max_new_tokens=512)
result = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])

# Show ground truth vs prediction
gt = json.loads(test_sample["ground_truth"])
gt_text = json.dumps(gt.get("gt_parse", {}), ensure_ascii=False, indent=2)

print("=== Ground Truth ===")
print(gt_text)
print("\n=== Model Prediction ===")
try:
    parsed = json.loads(result)
    print(json.dumps(parsed, ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print(result)